# A_FIS Test - Multidimensional Case

### IMPORTS

In [1]:
# Setup path and imports
import sys
from pathlib import Path

# Add AFIS_Repo root to path
REPO_ROOT = Path.cwd().parent.parent.parent
sys.path.insert(0, str(REPO_ROOT))

# Import from afis package
from afis.core.A_FIS import format_FN_N_Dim
from afis.core.afis_utils import FuzzySet, Trapezoidal
from afis.visualization.plotting import (
    create_rule_base, run_afis, plot_results, show_svi_table,
    show_detailed_diagnostic_nd, compute_activation_curves,
    plot_activation_curves
)

print("Loaded!")

Loaded!


### 3D DOMAIN EXAMPLE


In [2]:
# Define membership functions
A1 = Trapezoidal(ini=0, top1=15, top2=15, end=30)
A2 = Trapezoidal(ini=40, top1=50, top2=50, end=60)
A3 = Trapezoidal(ini=80, top1=100, top2=100, end=100)

B1 = Trapezoidal(ini=0, top1=0, top2=0, end=60)
B2 = Trapezoidal(ini=130, top1=170, top2=200, end=200)

C1 = Trapezoidal(ini=0, top1=0, top2=0, end=60)
C2 = Trapezoidal(ini=70, top1=80, top2=90, end=110)
C3 = Trapezoidal(ini=130, top1=150, top2=150, end=150)

D1 = Trapezoidal(ini=20, top1=50, top2=50, end=70)
D2 = Trapezoidal(ini=120, top1=150, top2=180, end=200)
D3 = Trapezoidal(ini=230, top1=270, top2=300, end=300)

# Create rule base
rule_base_3d = create_rule_base(
    antecedents=[
        [FuzzySet("A1", A1), FuzzySet("A2", A2), FuzzySet("A3", A3)],
        [FuzzySet("B1", B1), FuzzySet("B2", B2)],
        [FuzzySet("C1", C1), FuzzySet("C2", C2), FuzzySet("C3", C3)]
    ],
    consequents=[FuzzySet("D1", D1), FuzzySet("D2", D2), FuzzySet("D3", D3)],
    input_ranges=[(0, 100), (0, 200), (0, 150)],
    output_range=(0, 300),
    rules=[
        ([0, 0, 0], 0),
        ([1, 0, 1], 1),
        ([2, 1, 2], 2),
        ([1, 0, 1], 2)
    ]
)
print("Rule base created!")

Rule base created!


In [3]:
# Define input
input1 = Trapezoidal(ini=35, top1=35, top2=35, end=35)
input2 = Trapezoidal(ini=70, top1=70, top2=70, end=70)
input3 = Trapezoidal(ini=85, top1=85, top2=85, end=85)

input_list_3d = [input1, input2, input3]
input_formatted_3d = format_FN_N_Dim([FuzzySet(f"In{i}", inp) for i, inp in enumerate(input_list_3d)], 'rule_antecedent')

# Run A_FIS
output_3d, U_3d, y_crisp_3d, _ = run_afis(input_formatted_3d, rule_base_3d)
print(f"Input: ({input1.ini}, {input2.ini}, {input3.ini})")
print(f"Output: {y_crisp_3d:.2f}")

# Show S_A table
show_svi_table(rule_base_3d, input_formatted_3d)

Input: (35, 70, 85)
Output: 201.04

Rule Base:
Rule       Antecedents                                        Consequent          
------------------------------------------------------------------------------------------
0          A1, B1, C1                                         D1                  
1          A2, B1, C2                                         D2                  
2          A3, B2, C3                                         D3                  
3          A2, B1, C2                                         D3                  

A-subsethood Measures:
Rule       S_A(dim1)     S_A(dim2)     S_A(dim3)     S_A (agg)      
-------------------------------------------------------------------
0          0.375000      0.470588      0.000000      0.281863       
1          0.514286      0.470588      1.000000      0.661625       
2          0.000000      0.247059      0.000000      0.082353       
3          0.514286      0.470588      1.000000      0.661625       


In [4]:
# Plot antecedents and inputs for each dimension + output
plot_results(rule_base_3d, input_list_3d, output_3d, U_3d, y_crisp_3d)

### 2D DOMAIN EXAMPLE


In [5]:

# Dimension 1: 3 trapezoidal antecedents
T1 = Trapezoidal(ini=40, top1=70, top2=70, end=100)
T2 = Trapezoidal(ini=160, top1=190, top2=190, end=220)
T3 = Trapezoidal(ini=230, top1=260, top2=260, end=290)

# Dimension 2: 3 trapezoidal antecedents
S1 = Trapezoidal(ini=20, top1=60, top2=60, end=100)
S2 = Trapezoidal(ini=120, top1=160, top2=160, end=200)
S3 = Trapezoidal(ini=210, top1=250, top2=250, end=290)

# Consequents
Y1 = Trapezoidal(ini=0, top1=0, top2=0, end=50)
Y2 = Trapezoidal(ini=0, top1=50, top2=50, end=100)
Y3 = Trapezoidal(ini=50, top1=100, top2=100, end=150)
Y4 = Trapezoidal(ini=100, top1=150, top2=150, end=150)

# Create 2D rule base with 5 rules (all 6 antecedents used at least once)
rule_base_2d = create_rule_base(
    antecedents=[
        [FuzzySet("T1", T1), FuzzySet("T2", T2), FuzzySet("T3", T3)],  # Dim 1
        [FuzzySet("S1", S1), FuzzySet("S2", S2), FuzzySet("S3", S3)]   # Dim 2
    ],
    consequents=[FuzzySet("Y1", Y1), FuzzySet("Y2", Y2), FuzzySet("Y3", Y3), FuzzySet("Y4", Y4)],
    input_ranges=[(0, 300), (0,300)],
    output_range=(0, 150),
    rules=[
        ([0, 0], 0),  # T1, S1 → Y1
        ([1, 1], 1),  # T2, S2 → Y2
        ([2, 2], 3),  # T3, S3 → Y4
        ([2, 0], 2),  # T3, S1 → Y3
        ([0, 2], 2),  # T1, S2 → Y2
    ]
)
print("2D Rule base created with 5 rules!")
print("  Rule 0: T1, S1 → Y1")
print("  Rule 1: T2, S2 → Y2")
print("  Rule 2: T3, S3 → Y4")
print("  Rule 3: T3, S1 → Y3")
print("  Rule 4: T1, S3 → Y3")

2D Rule base created with 5 rules!
  Rule 0: T1, S1 → Y1
  Rule 1: T2, S2 → Y2
  Rule 2: T3, S3 → Y4
  Rule 3: T3, S1 → Y3
  Rule 4: T1, S3 → Y3


In [6]:
# Define multiple inputs for visualization
input1 =  [
    Trapezoidal(ini=60, top1=60, top2=60, end=60),   # X1 dim1
    Trapezoidal(ini=50, top1=50, top2=50, end=50) # X1 dim2
]
input2 = [
    Trapezoidal(ini=220, top1=220, top2=220, end=220),  # X2 dim1 (singleton)
    Trapezoidal(ini=90, top1=90, top2=90, end=90)   # X2 dim2 (singleton)
]
input3 = [
    Trapezoidal(ini=150, top1=150, top2=150, end=150),  # X3 dim1
    Trapezoidal(ini=250, top1=250, top2=250, end=250)       # X3 dim2
]

input4 = [
    Trapezoidal(ini=100, top1=100, top2=100, end=100),  # X3 dim1
    Trapezoidal(ini=150, top1=150, top2=150, end=150)       # X3 dim2
]

i1= 100  #100  #70
u2=  220  #220  #235
input4 = [
    Trapezoidal(ini=i1, top1=i1, top2=i1, end=i1),  # X3 dim1
    Trapezoidal(ini=u2, top1=u2, top2=u2, end=u2)       # X3 dim2
]



In [7]:
# Now we apply the A_FIS in this 2D rule base
# Now we apply the A_FIS in this 2D rule base
input_list = input4  # Add this line
input_formatted = format_FN_N_Dim([FuzzySet(f"In{i}", inp) for i, inp in enumerate(input_list)], 'rule_antecedent')
# output_2D, U, y_crisp, _ = run_afis(input_formatted, rule_base_2d,t_norm_type=['hamacher',0.2],imp_params=['hamacher', 0.2])
output_2D, U, y_crisp, _ = run_afis(input_formatted, rule_base_2d,t_norm_type='min',imp_params=['hamacher', 0.2])

# we visualize the output
plot_results(rule_base_2d, input_list, output_2D, U, y_crisp)

